In [92]:
import pandas as pd
import os

In [93]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [94]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [95]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [96]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [97]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [98]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [99]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [100]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [101]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [102]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [103]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [104]:
df = df.fillna({
    'release_year': -1
})

In [105]:
#  ne garder que les lignes qui ont un runtime supérieur à 44.0
df = df[df["runtime"] > 44.0]

In [106]:
# enlever les films qui ont uniquement la valeur 'Documentary' dans la colonne genres
df = df[~df["genres_array"].apply(lambda x: len(x) == 1 and x[0] == "Documentary")]

In [107]:
df.shape

(45355, 18)

In [108]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [109]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                     2
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                     78
release_year                     0
genres_array                    31
production_countries_array     568
production_companies_array    1856
cast_array                      50
director_array                  78
writers_array                 1009
Name: empty_count, dtype: int64


In [110]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [111]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [112]:
df.to_csv(clean_path, index=False)

## ML

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
# from scipy.sparse import hstack
import umap.umap_ as umap
import numpy as np

In [114]:
# Vectorisation TF-IDF sur les résumés
# faire des tests de temps en temps avec max_features=1000
vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
X_tfidf = vectorizer.fit_transform(df["overview"])

In [115]:
# Réduction dimensionnelle avec UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
embedding = reducer.fit_transform(X_tfidf.toarray())

df["x"] = embedding[:,0]
df["y"] = embedding[:,1]

/usr/local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [116]:
# def top_k_binarize(series, top_k=100):
#     # Compte la fréquence de chaque item
#     all_items = [item for sublist in series for item in sublist]
#     uniques, counts = np.unique(all_items, return_counts=True)
#     top_items = set([u for u, c in sorted(zip(uniques, counts), key=lambda x: -x[1])[:top_k]])

#     # Filtre et binarize
#     filtered_series = [[i for i in lst if i in top_items] for lst in series]
#     mlb = MultiLabelBinarizer(classes=list(top_items))
#     return mlb.fit_transform(filtered_series)

In [117]:
# genres_matrix = top_k_binarize(df['genres_array'], top_k=50)
# cast_matrix = top_k_binarize(df['cast_array'], top_k=500)
# director_matrix = top_k_binarize(df['director_array'], top_k=200)
# writers_matrix = top_k_binarize(df['writers_array'], top_k=200)
# production_companies_matrix = top_k_binarize(df['production_companies_array'], top_k=100)
# production_countries_matrix = top_k_binarize(df['production_countries_array'], top_k=50)

In [118]:
# scaler = StandardScaler()
# year_matrix = scaler.fit_transform(df[['release_year']])

In [119]:
# # Concaténer tout dans un espace euclidien
# features = np.hstack([
#     embedding * 2.0,                # TF-IDF / UMAP de l’overview → très important pour le contenu
#     genres_matrix * 1.5,            # Genres → assez important, mais moins que le texte
#     production_countries_matrix * 0.3,  # Pays → influence plus faible
#     production_companies_matrix * 0.3,  # Société de prod → faible
#     cast_matrix * 0.8,              # Cast → modérément important
#     director_matrix * 1.0,          # Réalisateur → important
#     writers_matrix * 0.5,           # Scénaristes → moins important que réalisateur/cast
#     year_matrix * 0.2               # Année de sortie → faible importance
# ])

In [120]:
def find_closest_movies(df, input_title, top_n=10):
    if input_title not in df['title'].values:
        print(f"Le film '{input_title}' n'a pas été trouvé.")
        return None
    
    input_point = df.loc[df['title'] == input_title, ['x','y']].values[0]
    df['distance'] = np.linalg.norm(df[['x','y']].values - input_point, axis=1)
    
    df_filtered = df[df['title'] != input_title]
    closest_movies = df_filtered.nsmallest(top_n, 'distance')
    
    return closest_movies[['title','distance']]

In [121]:
closest_10 = find_closest_movies(df, "Schindler's List", top_n=10)
print(closest_10)

                                   title  distance
337499                        Resistance  0.027908
13644                 The Rape of Europa  0.042668
289414  David Bowie: The Last Five Years  0.044362
3878                  The Counterfeiters  0.048872
2756                        Never So Few  0.059654
45249                   Love and Anarchy  0.060207
31633                            Korczak  0.079598
4427                          Black Book  0.086423
13353                  Reach for the Sky  0.086840
47790                         Invincible  0.088718


In [122]:
# def recommend_hybrid(df, input_title, features, top_n=10):
#     if input_title not in df['title'].values:
#         print(f"Le film '{input_title}' n'a pas été trouvé.")
#         return None
    
#     # Coordonnées du film
#     input_idx = df[df['title'] == input_title].index[0]
#     input_vector = features[input_idx]

#     # Distances euclidiennes
#     distances = np.linalg.norm(features - input_vector, axis=1)

#     # Exclure le film lui-même
#     df['distance'] = distances
#     df_filtered = df[df['title'] != input_title]

#     # Récupérer les top_n les plus proches
#     closest = df_filtered.nsmallest(top_n, 'distance')
#     return closest[['title','distance']]

In [123]:
# closest_movies = recommend_hybrid(df, "Schindler's List", features, top_n=10)
# print(closest_movies)